# Prediction

In [11]:
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load trained artifacts
model = load_model("model_v2.h5")

with open("tokenizer_v2.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("config_v2.pkl", "rb") as f:
    config = pickle.load(f)

max_len = config["max_len"]
print(f"Model loaded. max_len = {max_len}")

max_len = config["max_len"]
num_classes = config["num_classes"]
id_to_answer = config["id_to_answer"]

Model loaded. max_len = 14


In [12]:
def generate_response(text: str, confidence_threshold: float = 0.25 ) -> tuple:
    
    seq = tokenizer.texts_to_sequences([text.lower()])
    ## seq = pad_sequences(seq, maxlen=max_len, padding='post')

    # pred = model.predict(seq, verbose=0)
    # pred = np.argmax(pred, axis=-1)[0]

    # index_to_word = {v: k for k, v in tokenizer.word_index.items()}

    # result = []
    # for idx in pred:
    #     if idx != 0:
    #         word = index_to_word.get(idx, "")
    #         if word:
    #             result.append(word)

    # return " ".join(result)
    padded = pad_sequences(seq, maxlen=max_len, padding= 'post')

    probs = model.predict(padded, verbose=0)[0]
    class_id = int(np.argmax(probs))
    confidence = float(probs[class_id])

    if confidence < confidence_threshold:
        return "I'm not sure about that. Please ask about Cambodia tourism topics.", confidence
    return id_to_answer[class_id],confidence



#### TEST 

In [13]:
test_cases = [
    "where is angkor wat specifically",
    "what food should i try in cambodia",
    "best time to visit cambodia",
    "is cambodia safe",
    "how to travel in phnom penh",
]

print("=" * 65)
for q in test_cases:
    answer, conf = generate_response(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"Confidence : {conf:.2%}")
    print("-" * 65)

Q: where is angkor wat specifically
A: angkor wat is in siem reap cambodia
Confidence : 97.68%
-----------------------------------------------------------------
Q: what food should i try in cambodia
A: you should try amok and lok lak
Confidence : 93.61%
-----------------------------------------------------------------
Q: best time to visit cambodia
A: november to february is the best time
Confidence : 83.11%
-----------------------------------------------------------------
Q: is cambodia safe
A: it is generally safe for tourists
Confidence : 67.10%
-----------------------------------------------------------------
Q: how to travel in phnom penh
A: you can use tuk tuk or taxi
Confidence : 78.18%
-----------------------------------------------------------------


TEST2

In [14]:
error_cases = [
    ("Angkor Wat location?",              "Rephrased / short-form question"),
    ("WHERE IS ANGKOR WAT",               "Uppercase input — tokenizer is case-sensitive"),
    ("What's the weather like?",          "Out-of-domain question"),
    ("population of siem reap",           "Fact not in training data"),
    ("Can you book a hotel for me?",      "Action the model cannot perform"),
]

print("=" * 70)
for q, reason in error_cases:
    output = generate_response(q)
    print(f"Input          : {q}")
    print(f"Model output   : {output}")
    print(f"Failure reason : {reason}")
    print("-" * 70)

Input          : Angkor Wat location?
Model output   : (np.str_('it is a famous temple in cambodia'), 0.28520169854164124)
Failure reason : Rephrased / short-form question
----------------------------------------------------------------------
Input          : WHERE IS ANGKOR WAT
Model output   : (np.str_('angkor wat is in siem reap cambodia'), 0.8231677412986755)
Failure reason : Uppercase input — tokenizer is case-sensitive
----------------------------------------------------------------------
Input          : What's the weather like?
Model output   : ("I'm not sure about that. Please ask about Cambodia tourism topics.", 0.21098250150680542)
Failure reason : Out-of-domain question
----------------------------------------------------------------------
Input          : population of siem reap
Model output   : ("I'm not sure about that. Please ask about Cambodia tourism topics.", 0.24169936776161194)
Failure reason : Fact not in training data
---------------------------------------------

## Summary of Error Analysis

| Failure Type | Root Cause |
|---|---|
| Rephrased question | Model matches surface patterns, not meaning |
| Uppercase input | `Tokenizer` is case-sensitive by default |
| Out-of-domain question | No matching pattern in training data |
| Missing fact | Dataset does not contain that information |
| Impossible task | Model generates text only; cannot take actions |

**Core limitation:** SimpleRNN memorises input→output mappings. It has no semantic understanding, so any variation in wording causes it to fail. The vanishing-gradient problem also limits how well it handles long sentences.